In [ ]:
!pip install -q tokenizers transformers tensorflow matplotlib seaborn koreanize-matplotlib

In [ ]:
# ============================================================
# 실습 1-2. Self-Attention 시각화
# 목표: 토큰 간 관계를 가중치 행렬로 계산하고 히트맵으로 확인한다
# 사전 설치: pip install tensorflow matplotlib seaborn koreanize-matplotlib
# ============================================================

import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
plt.rcParams['axes.unicode_minus'] = False

# ============================================================
# [실습 변수] 여기만 수정하여 다양한 실험 가능
# ============================================================

# 시각화에 사용할 토큰 목록 (문장을 단어 단위로 분리한 것)
# 실제 모델에서는 BPE 토큰이 들어오지만 여기서는 이해를 위해 단어 사용
TOKENS = ["어텐션", "가중치는", "토큰", "간", "관련성을", "나타낸다"]

# 임베딩 차원: 각 토큰을 몇 차원 벡터로 표현할지
# 실제 GPT-3는 12288, BERT-base는 768 사용
# 여기서는 시각화 목적이므로 작게 설정
D_MODEL = 16

# 랜덤 시드: 같은 값이면 실행할 때마다 동일한 결과
# 다른 숫자로 바꾸면 가중치 패턴이 달라짐
RANDOM_SEED = 42

# 히트맵 색상 테마
# 'YlOrRd'(노랑-주황-빨강), 'Blues', 'Greens', 'RdBu' 중 선택
HEATMAP_CMAP = "YlOrRd"

# 막대 그래프로 집중도를 볼 기준 토큰 (TOKENS 리스트의 인덱스)
# 0=첫번째 토큰, 1=두번째 토큰 ...
FOCUS_TOKEN_IDX = 0

# ============================================================
# 이하 코드는 수정 불필요
# ============================================================

# ── Scaled Dot-Product Attention 구현 ────────────────────────
# Attention 공식: softmax(Q * K^T / sqrt(d_k)) * V
#
# Q (Query) : "나는 무엇을 찾고 있나?" 를 나타내는 벡터
# K (Key)   : "나는 어떤 정보를 가지고 있나?" 를 나타내는 벡터
# V (Value) : "실제로 전달할 정보" 벡터
# sqrt(d_k) : 차원이 커질수록 내적값이 커지는 문제를 완화하기 위한 스케일링

class ScaledDotProductAttention(keras.layers.Layer):
    def call(self, q, k, v, mask=None):
        d_k = tf.cast(tf.shape(k)[-1], tf.float32)

        # Q와 K의 내적: 두 벡터가 비슷할수록 높은 점수
        scores = tf.matmul(q, k, transpose_b=True) / tf.math.sqrt(d_k)

        if mask is not None:
            # 마스킹: 참조하면 안 되는 위치를 음의 무한대로 설정
            # softmax 통과 시 해당 위치의 가중치가 0에 가까워짐
            scores += (mask * -1e9)

        # softmax: 점수를 0~1 사이 확률값으로 변환 (행 합계 = 1.0)
        weights = tf.nn.softmax(scores, axis=-1)
        output  = tf.matmul(weights, v)
        return output, weights


class SelfAttentionLayer(keras.layers.Layer):
    """Self-Attention: Q, K, V 모두 같은 입력에서 생성"""
    def __init__(self, d_model, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        # 세 개의 가중치 행렬로 입력을 Q, K, V로 변환
        self.wq   = keras.layers.Dense(d_model, use_bias=False, name="W_Q")
        self.wk   = keras.layers.Dense(d_model, use_bias=False, name="W_K")
        self.wv   = keras.layers.Dense(d_model, use_bias=False, name="W_V")
        self.attn = ScaledDotProductAttention()

    def call(self, x, mask=None):
        q = self.wq(x)  # 입력을 Query로 변환
        k = self.wk(x)  # 입력을 Key로 변환
        v = self.wv(x)  # 입력을 Value로 변환
        out, weights = self.attn(q, k, v, mask)
        return out, weights


# ── Self-Attention 실행 ───────────────────────────────────────
tf.random.set_seed(RANDOM_SEED)

seq_len = len(TOKENS)
# 랜덤 임베딩: 실제로는 학습된 임베딩을 사용하지만
# 여기서는 Attention 구조를 보여주기 위해 랜덤 입력 사용
x = tf.random.normal([1, seq_len, D_MODEL])
# shape: (batch=1, 시퀀스 길이, 임베딩 차원)

self_attn = SelfAttentionLayer(d_model=D_MODEL)
output, attn_weights = self_attn(x)

# Attention 가중치 행렬 추출: (seq_len, seq_len) 형태
weights_np = attn_weights.numpy()[0]

print(f"입력 shape  : {x.shape}  -> (batch, seq_len, d_model)")
print(f"출력 shape  : {output.shape}")
print(f"가중치 shape: {weights_np.shape}  -> (seq_len, seq_len)")
print(f"\n가중치 행렬 (각 행의 합 = 1.0):")
for i, token in enumerate(TOKENS):
    row = " ".join(f"{w:.2f}" for w in weights_np[i])
    print(f"  {token:8s}: {row}")

# ── 히트맵 시각화 ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 전체 Attention 가중치 히트맵
sns.heatmap(
    weights_np,
    xticklabels=TOKENS,
    yticklabels=TOKENS,
    annot=True, fmt=".2f",
    cmap=HEATMAP_CMAP,
    linewidths=0.5,
    ax=axes[0]
)
axes[0].set_title(
    "Self-Attention 가중치\n(행: Query 토큰, 열: Key 토큰)",
    fontsize=12
)
axes[0].set_xlabel("Key (참조 대상)")
axes[0].set_ylabel("Query (현재 토큰)")
axes[0].tick_params(axis='x', rotation=30)

# 오른쪽: 특정 토큰의 집중 분포 막대 그래프
focus_token  = TOKENS[FOCUS_TOKEN_IDX]
focus_weights = weights_np[FOCUS_TOKEN_IDX]

axes[1].bar(TOKENS, focus_weights, color='steelblue', alpha=0.8)
axes[1].set_title(
    f'"{focus_token}" 토큰이\n각 토큰에 집중하는 정도',
    fontsize=12
)
axes[1].set_ylabel("Attention 가중치")
axes[1].tick_params(axis='x', rotation=30)
axes[1].set_ylim(0, 1.0)
# 가중치 값 표시
for i, v in enumerate(focus_weights):
    axes[1].text(i, v + 0.02, f"{v:.2f}", ha='center', fontsize=9)

plt.suptitle("Scaled Dot-Product Self-Attention 시각화", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("attention_visualization.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n읽는 법: 행(Query 토큰)이 열(Key 토큰)을 얼마나 참고하는지를 색 농도로 표현")
print("RAG 연결: 질문 토큰이 문서 토큰 중 어느 부분에 집중하는지 - 같은 원리")

# ── 추가 실습: FOCUS_TOKEN_IDX 변경 실험 ────────────────────
print("\n" + "=" * 50)
print("추가 확인: 토큰별 최고 집중 대상")
print("=" * 50)
for i, token in enumerate(TOKENS):
    max_idx = np.argmax(weights_np[i])
    print(f"  '{token}' -> 가장 집중: '{TOKENS[max_idx]}' ({weights_np[i][max_idx]:.2f})")